# 03 - Non-Working Day PRESENT Analysis

## Objective


This notebook investigates records where employees are marked as PRESENT on non-working days such as WEEK-OFF and HOLIDAY.

The objective is to understand:

- How many such records exist
- Whether these records have actual TIC/TOF/TOTAL hours
- Which locations show higher counts and rates
- Whether the pattern represents full-day work or short-duration presence
- What business implications may exist

## Important Granularity Note

This analysis is performed at the record level, not necessarily at the unique employee level.

Since employees may work on multiple projects, the same `EMP_CODE` and `ATTENDANCE_DATE` can have multiple valid records. Therefore, counts in this notebook should be interpreted as record counts unless unique employee counts are explicitly calculated.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

project_root = Path.cwd().parent
processed_path = project_root / "data" / "processed"

no_duplicate_file_path = processed_path / "combined_no_exact_duplicates.csv"

df = pd.read_csv(no_duplicate_file_path, low_memory=False)

print("Data loaded successfully")
print("Shape:", df.shape)

In [ ]:
non_working_present = df[
    (df["WEEKOFF_HOLIDAY"].isin(["WEEK-OFF", "HOLIDAY"])) &
    (df["REMARKS"] == "PRESENT")
]

print("Non-working day PRESENT records:", non_working_present.shape[0])

non_working_present[
    ["TIC_HOURS_NEW", "TOF_HOURS_NEW", "RODC_HOURS", "ALLODC_HOURS", "TOTAL_HOURS"]
].describe()

A total of **62,969 records** are marked `PRESENT` on non-working days such as `WEEK-OFF` or `HOLIDAY`.

These records contain measurable campus and ODC/floor hours:

- Average TIC hours: approximately **4.52 hours**
- Median TIC hours: approximately **2.71 hours**
- 75th percentile TIC hours: approximately **8.27 hours**
- Maximum TIC hours: approximately **42.33 hours**

This indicates that non-working day `PRESENT` records are not only status labels; many of them also have actual time recorded.

This may represent valid weekend/holiday work due to project, client, shift, or operational requirements. However, very high hour values such as TIC above 40 hours require further validation as possible data quality, overnight shift, or multi-project aggregation cases.

In [ ]:
non_working_present_location = (
    non_working_present["LOCATION"]
    .value_counts()
    .reset_index()
)

non_working_present_location.columns = ["LOCATION", "non_working_present_count"]

non_working_present_location.head(15)

## Location Pattern of Non-Working Day PRESENT Records

The highest absolute number of non-working day `PRESENT` records appears in larger locations such as Bengaluru, Pune, Hyderabad, Mumbai, Kolkata, and Chennai.

However, raw counts alone can be misleading because larger locations naturally have more total records.

Therefore, location-level comparison should consider both:

- Absolute count
- Normalized rate

This helps distinguish between locations with high total volume and locations where the non-working day `PRESENT` pattern is relatively stronger.

### Initial Rate Using Total Location Records

In [ ]:
location_total_records = (
    df["LOCATION"]
    .value_counts()
    .reset_index()
)

location_total_records.columns = ["LOCATION", "total_records"]

location_pattern = non_working_present_location.merge(
    location_total_records,
    on="LOCATION",
    how="left"
)

location_pattern["non_working_present_rate_pct"] = (
    location_pattern["non_working_present_count"] /
    location_pattern["total_records"] * 100
)

location_pattern.sort_values(
    "non_working_present_rate_pct",
    ascending=False
).head(15)

Although Bengaluru has the highest absolute number of non-working day PRESENT records, Mumbai shows a higher relative rate when normalized by total records.

Bengaluru has 19,265 such records out of 1,642,661 total records, resulting in a rate of 1.17%.

Mumbai has 6,381 such records out of 446,494 total records, resulting in a rate of 1.43%.

This suggests that Mumbai has a relatively stronger non-working day presence pattern than Bengaluru, despite having fewer absolute records.

However, very small locations such as Cebu and Kuala Lumpur show high percentages due to very small record counts, so they should be interpreted cautiously.

This initial rate gives a broad location-level comparison. However, since the pattern specifically relates to non-working days, a more appropriate denominator is total non-working records per location. Therefore, the next section calculates the preferred rate using only WEEK-OFF and HOLIDAY records.

In [ ]:
# Total non-working records by location
non_working_records = df[
    df["WEEKOFF_HOLIDAY"].isin(["WEEK-OFF", "HOLIDAY"])
]

location_non_working_total = (
    non_working_records["LOCATION"]
    .value_counts()
    .reset_index()
)

location_non_working_total.columns = ["LOCATION", "total_non_working_records"]


# PRESENT records on non-working days by location
location_non_working_present = (
    non_working_present["LOCATION"]
    .value_counts()
    .reset_index()
)

location_non_working_present.columns = ["LOCATION", "non_working_present_count"]


# Merge both
location_non_working_rate = location_non_working_present.merge(
    location_non_working_total,
    on="LOCATION",
    how="left"
)

# Calculate rate
location_non_working_rate["non_working_present_rate_pct"] = (
    location_non_working_rate["non_working_present_count"] /
    location_non_working_rate["total_non_working_records"] * 100
)

# Sort by rate
location_non_working_rate.sort_values(
    "non_working_present_rate_pct",
    ascending=False
).head(20)

### Location-wise Non-working Day PRESENT Rate

To avoid misleading conclusions from raw counts, non-working day PRESENT records were normalized against total non-working records per location.

The rate was calculated as:

Non-working PRESENT Rate = Non-working PRESENT Count / Total WEEK-OFF + HOLIDAY Records

Key observations:

- Bengaluru has the highest absolute count of non-working day PRESENT records: 19,265.
- However, Mumbai has a higher normalized rate than Bengaluru:
  - Mumbai: 6,381 / 139,288 = 4.58%
  - Bengaluru: 19,265 / 502,487 = 3.83%
- Mysore shows a notably high non-working day PRESENT rate:
  - Mysore: 1,666 / 14,193 = 11.74%

This suggests that while Bengaluru has the largest volume, locations like Mysore and Mumbai show stronger relative non-working day presence patterns.

Small locations such as Cebu, Kuala Lumpur, and Rourkela show high rates but have very small record counts, so their percentages should be interpreted cautiously.

This is the preferred rate for this analysis because the numerator and denominator both relate specifically to non-working day records.

## Target Locations for Deeper Hour Analysis

Based on both absolute count and normalized non-working day PRESENT rate, the following locations were selected for deeper analysis:

- **Bengaluru**: highest absolute count
- **Mysore**: highest meaningful normalized rate
- **Mumbai**: strong rate with meaningful volume
- **Coimbatore**: relatively high rate with decent volume
- **Kolkata**: high volume with meaningful rate

Small locations such as Cebu and Kuala Lumpur were not selected for deeper analysis despite high percentages because their record counts are too small and may lead to unstable conclusions.

## Hour Pattern Analysis for Target Locations

The next step is to compare TIC, TOF, and TOTAL hours across selected target locations.

This helps determine whether non-working day PRESENT records represent:

- Full or near full working shifts
- Partial/mixed presence
- Short-duration visits

In [ ]:
target_locations = ["MYSORE", "MUMBAI", "COIMBATORE", "BENGALURU", "KOLKATA"]

target_data = non_working_present[
    non_working_present["LOCATION"].isin(target_locations)
]

hours_summary = (
    target_data
    .groupby("LOCATION")[["TIC_HOURS_NEW", "TOF_HOURS_NEW", "TOTAL_HOURS"]]
    .describe()
)

hours_summary


## Key Insight: Non-Working Day PRESENT Pattern Across Locations

### Finding

A meaningful pattern was identified where records are marked as `PRESENT` on non-working days such as `WEEK-OFF` and `HOLIDAY`.

This pattern is important because these records are not only status labels; many of them also contain actual TIC, TOF, and TOTAL hour values.

---

### Supporting Evidence

A total of **62,969 records** were marked as `PRESENT` on non-working days.

These records show measurable campus and ODC/floor presence:

- Average TIC hours: approximately **4.52 hours**
- Median TIC hours: approximately **2.71 hours**
- 75th percentile TIC hours: approximately **8.27 hours**
- Maximum TIC hours: approximately **42.33 hours**

This suggests that a portion of non-working day PRESENT records represent actual time spent on campus or in ODC/floor areas.

---

### Location-Level Pattern

Location-wise analysis shows that the pattern varies significantly across locations.

#### Absolute Count View

- Bengaluru has the highest absolute count of non-working day PRESENT records.
- Other high-volume locations include Pune, Hyderabad, Mumbai, Kolkata, Chennai, and Coimbatore.

However, raw counts alone can be misleading because larger locations naturally have more total records.

---

### Normalized Rate View

To compare locations more fairly, non-working day PRESENT records were normalized against total non-working records for each location.

The rate was calculated as:

`Non-working PRESENT Rate = Non-working PRESENT Count / Total WEEK-OFF + HOLIDAY Records`

Key observations:

- **Bengaluru** has the highest absolute count:
  - 19,265 PRESENT records out of 502,487 non-working records
  - Rate: **3.83%**

- **Mumbai** has a lower absolute count but a higher normalized rate:
  - 6,381 PRESENT records out of 139,288 non-working records
  - Rate: **4.58%**

- **Mysore** shows the highest meaningful normalized rate:
  - 1,666 PRESENT records out of 14,193 non-working records
  - Rate: **11.74%**

Small locations such as Cebu, Kuala Lumpur, and Rourkela show high rates but have very small record counts, so their percentages should be interpreted cautiously due to the small denominator effect.

---

### Hour Pattern Insights by Target Location

A deeper hour-based comparison was performed for the following target locations:

- Mysore
- Mumbai
- Coimbatore
- Bengaluru
- Kolkata

The hour pattern shows three distinct behavior types.

---

#### 1. Full / Near Full Working Pattern: Mumbai and Kolkata

Mumbai and Kolkata show relatively high TIC hour values.

- Mumbai:
  - Mean TIC: approximately **6.30 hours**
  - Median TIC: approximately **6.45 hours**
  - 75th percentile TIC: approximately **9.81 hours**

- Kolkata:
  - Mean TIC: approximately **6.39 hours**
  - Median TIC: approximately **7.32 hours**
  - 75th percentile TIC: approximately **10.19 hours**

This suggests that non-working day PRESENT records in Mumbai and Kolkata are often associated with substantial campus presence and may represent full or near full working shifts.

---

#### 2. Partial / Mixed Presence Pattern: Bengaluru and Coimbatore

Bengaluru and Coimbatore show moderate average TIC hours with lower medians.

- Bengaluru:
  - Mean TIC: approximately **3.56 hours**
  - Median TIC: approximately **2.00 hours**
  - 75th percentile TIC: approximately **4.94 hours**

- Coimbatore:
  - Mean TIC: approximately **3.96 hours**
  - Median TIC: approximately **1.80 hours**
  - 75th percentile TIC: approximately **6.86 hours**

This suggests a mixed pattern where some records represent longer working presence, while many others appear to be shorter-duration visits.

---

#### 3. Short-Duration Presence Pattern: Mysore

Mysore shows a high non-working day PRESENT rate but very low TIC hour values.

- Mean TIC: approximately **1.61 hours**
- Median TIC: approximately **0.40 hours**
- 75th percentile TIC: approximately **1.39 hours**

This suggests that Mysore’s high non-working day PRESENT rate is mostly driven by short-duration presence rather than full-day work.

This pattern should be investigated further before drawing business conclusions.

---

### Business Interpretation

The non-working day PRESENT pattern should not be directly treated as a policy violation or anomaly.

It may represent:

- Weekend or holiday project work
- Client support requirements
- Shift-based operations
- Deployment or maintenance activity
- Business-critical work
- Multi-project allocation effects
- Data quality or classification issues

---

### Business Implications

#### Positive Signal

Non-working day presence may indicate operational commitment and business continuity, especially in locations such as Mumbai and Kolkata where the recorded TIC hours suggest substantial campus presence.

#### Potential Workforce Risk

If full-day work on week-offs or holidays is repeated frequently, it may create long-term workforce risks such as:

- Reduced work-life balance
- Employee fatigue
- Motivation decline
- Burnout risk
- Potential attrition risk

#### Data Quality / Validation Need

Very high hour values, such as TIC or TOTAL hours greater than 24, may require separate validation because they could be caused by:

- Multi-project allocation
- Overnight shifts
- System calculation issues
- Repeated allocation records
- Data aggregation effects

---

### Conclusion

Non-working day PRESENT records represent a meaningful operational pattern.

The pattern is not uniform across locations:

- Bengaluru has the highest absolute volume.
- Mumbai and Kolkata show stronger full-day or near full-day working patterns.
- Mysore shows high relative frequency but mostly short-duration presence.
- Coimbatore and Bengaluru show mixed-duration behavior.

This pattern should be explored further by checking employee-level repetition, career band, billability status, high-hour cases, and business context before making final conclusions.


## Next Analysis Direction

The next analysis can focus on:

1. High-hour non-working day records where TIC or TOTAL hours exceed 24.
2. Repeated non-working day presence at employee level.
3. Career band and billability-wise distribution.
4. Workday patterns such as PRESENT, Home, Client Office, Leave, and Wipro Office.

## Important Note Before High-Hour Analysis

PM clarified that:

- `TOTAL_HOURS = RODC_HOURS + ALLODC_HOURS`
- `TIC_HOURS_NEW` represents Time In Campus
- `TOF_HOURS_NEW` represents Time On Floor / ODC floor time
- Employees may work on multiple projects, so the same `EMP_CODE` and `ATTENDANCE_DATE` can have multiple valid records

Therefore, high hour values should be interpreted carefully.

At this stage, high-hour records are analyzed at the record level only. A high value such as TIC or TOTAL greater than 24 hours should not be directly interpreted as confirmed continuous working time.

Such records are treated as review cases because they may be caused by:

- Multi-project allocation effects
- Overnight shift logic
- System calculation behavior
- Repeated allocation records
- Genuine exceptional support or project work

The purpose of this section is to identify and quantify high-hour records for further validation, not to make final conclusions.

In [ ]:
# Define high-hour threshold
high_hour_threshold = 24

# Identify high-hour records within non-working day PRESENT records
high_hour_non_working_present = non_working_present[
    (non_working_present["TIC_HOURS_NEW"] > high_hour_threshold) |
    (non_working_present["TOF_HOURS_NEW"] > high_hour_threshold) |
    (non_working_present["TOTAL_HOURS"] > high_hour_threshold)
]

print("Total non-working day PRESENT records:", non_working_present.shape[0])
print("High-hour non-working day PRESENT records:", high_hour_non_working_present.shape[0])

high_hour_rate = (
    high_hour_non_working_present.shape[0] /
    non_working_present.shape[0] * 100
)

print("High-hour record percentage:", round(high_hour_rate, 2), "%")

high_hour_non_working_present[
    [
        "EMP_CODE",
        "ATTENDANCE_DATE",
        "LOCATION",
        "WEEKOFF_HOLIDAY",
        "REMARKS",
        "TIC_HOURS_NEW",
        "TOF_HOURS_NEW",
        "RODC_HOURS",
        "ALLODC_HOURS",
        "TOTAL_HOURS",
        "CAREER_BAND",
        "BILLABILITY_STATUS",
        "SOURCE_FILE"
    ]
].head(20)

## High-Hour Cases Interpretation

Out of **62,969** non-working day `PRESENT` records, **446 records** have unusually high hour values where TIC, TOF, or TOTAL hours exceed 24 hours.

This represents only **0.71%** of the non-working day PRESENT records.

This suggests that extreme high-hour values exist, but they form a very small part of the overall non-working day presence pattern. Therefore, the broader non-working day PRESENT pattern is not primarily driven by these extreme hour records.

However, these high-hour records should be treated as review cases because values above 24 hours may be affected by:

- Multi-project allocation effects
- Overnight shift logic
- System calculation behavior
- Repeated allocation records
- Exceptional support or project work

At this stage, these should not be interpreted as confirmed continuous working hours. They require further validation before drawing final conclusions.

High-hour records are rare, so they do not appear to be the main driver of the non-working day PRESENT pattern.

But also:

High-hour records should be separately reviewed because they may indicate data quality, aggregation, or exceptional work scenarios.

In [ ]:
high_hour_location_summary = (
    high_hour_non_working_present["LOCATION"]
    .value_counts()
    .reset_index()
)

high_hour_location_summary.columns = ["LOCATION", "high_hour_record_count"]

high_hour_location_summary.head(20)

High-hour review cases are concentrated mostly in larger locations such as Mumbai, Bengaluru, Hyderabad, Pune, Coimbatore, and Kolkata.

Mumbai and Bengaluru have the highest absolute number of high-hour non-working day PRESENT records.

Mysore, despite having a high non-working day PRESENT rate, has only 1 high-hour record. This supports the earlier observation that Mysore’s pattern is primarily short-duration presence rather than full-day or high-hour work.

However, raw high-hour counts should be normalized against total non-working day PRESENT records per location before drawing stronger conclusions.

In [ ]:
high_hour_location_rate = high_hour_location_summary.merge(
    location_non_working_present,
    on="LOCATION",
    how="left"
)

high_hour_location_rate["high_hour_rate_within_non_working_present_pct"] = (
    high_hour_location_rate["high_hour_record_count"] /
    high_hour_location_rate["non_working_present_count"] * 100
)

high_hour_location_rate.sort_values(
    "high_hour_rate_within_non_working_present_pct",
    ascending=False
).head(20)

## High-Hour Cases by Location

Among the 446 high-hour non-working day PRESENT records, Mumbai and Bengaluru have the highest absolute counts:

- Mumbai: 102 high-hour records
- Bengaluru: 101 high-hour records
- Hyderabad: 59 high-hour records
- Pune: 57 high-hour records
- Coimbatore: 39 high-hour records
- Kolkata: 36 high-hour records

However, after normalizing high-hour records against total non-working day PRESENT records in each location, the pattern changes.

Mumbai has both a high absolute count and a relatively higher high-hour rate:

- Mumbai: 102 / 6,381 = 1.60%
- Bengaluru: 101 / 19,265 = 0.52%

This suggests that high-hour review cases are proportionally more prominent in Mumbai than Bengaluru, even though both locations have almost the same absolute high-hour count.

Mysore has only 1 high-hour record out of 1,666 non-working day PRESENT records, resulting in a very low high-hour rate of 0.06%. This supports the earlier observation that Mysore's non-working day PRESENT pattern is mostly short-duration presence rather than full-day or high-hour work.

Overall, high-hour cases are rare, representing only 0.71% of all non-working day PRESENT records. Therefore, they do not appear to be the main driver of the overall pattern, but they should still be treated as review cases for validation.

## Employee Repetition in Non-Working Day PRESENT Records

This section checks whether non-working day PRESENT records are spread across many employees or concentrated among a smaller group of employees who repeatedly appear on week-offs or holidays.

The purpose is to identify whether this is mostly an occasional pattern or a repeated employee-level pattern that may indicate workload concentration, support dependency, or review-worthy operational behavior.


In [ ]:
employee_repetition = (
    non_working_present
    .groupby("EMP_CODE")
    .agg(
        non_working_present_records=("ATTENDANCE_DATE", "count"),
        unique_non_working_present_dates=("ATTENDANCE_DATE", "nunique"),
        avg_tic_hours=("TIC_HOURS_NEW", "mean"),
        max_tic_hours=("TIC_HOURS_NEW", "max"),
        avg_total_hours=("TOTAL_HOURS", "mean"),
        max_total_hours=("TOTAL_HOURS", "max")
    )
    .reset_index()
    .sort_values("unique_non_working_present_dates", ascending=False)
)

print("Unique employees with non-working day PRESENT records:", employee_repetition.shape[0])

employee_repetition.head(20)


## Employee Repetition Analysis

The analysis shows that non-working day PRESENT records are not only isolated one-time cases. Some employees appear repeatedly across multiple WEEK-OFF or HOLIDAY dates.

For example, the top repeated employee appears on 25 unique non-working dates. Several other employees appear on 18–19 unique non-working dates.

This suggests that non-working day presence may be concentrated among certain employees rather than being completely random across the workforce.

However, the intensity varies:

- Some repeated employees have high average TIC hours, suggesting full or near full working presence on off-days.
- Some repeated employees have low average TIC hours, suggesting short-duration or partial presence.

This pattern should be interpreted carefully and validated with business context such as support roles, project requirements, shift schedules, or approved weekend work.

## Employee Frequency-Intensity Segmentation

After identifying repeated non-working day PRESENT records, the next step is to segment employees based on two dimensions:

1. **Frequency**  
   Number of unique WEEK-OFF/HOLIDAY dates where the employee was marked PRESENT.

2. **Intensity**  
   Average TIC hours during non-working day PRESENT records.

For exploratory segmentation:

- High Frequency = 5 or more unique non-working PRESENT dates
- High Intensity = Average TIC hours greater than or equal to 6 hours

These thresholds are used for exploratory pattern discovery and can be adjusted after business validation.

The purpose of this segmentation is to distinguish between:

- Occasional short visits
- Occasional full-day off-day work
- Repeated short-duration presence
- Repeated long-duration non-working day presence

The most critical segment is High Frequency + High Intensity because it may indicate repeated off-day workload, operational dependency, or potential work-life balance risk.

In [ ]:
frequency_threshold = 5
intensity_threshold = 6

employee_repetition["frequency_segment"] = np.where(
    employee_repetition["unique_non_working_present_dates"] >= frequency_threshold,
    "High Frequency",
    "Low Frequency"
)

employee_repetition["intensity_segment"] = np.where(
    employee_repetition["avg_tic_hours"] >= intensity_threshold,
    "High Intensity",
    "Low Intensity"
)

employee_repetition["employee_presence_segment"] = (
    employee_repetition["frequency_segment"] + " + " + employee_repetition["intensity_segment"]
)

segment_summary = (
    employee_repetition["employee_presence_segment"]
    .value_counts()
    .reset_index()
)

segment_summary.columns = ["employee_presence_segment", "employee_count"]

segment_summary

## Employee Frequency-Intensity Segmentation Result

Employees with non-working day PRESENT records were segmented using two dimensions:

- **Frequency**: Number of unique WEEK-OFF/HOLIDAY dates where the employee was marked PRESENT.
- **Intensity**: Average TIC hours during non-working day PRESENT records.

Exploratory thresholds used:

- High Frequency = 5 or more unique non-working PRESENT dates
- High Intensity = Average TIC hours >= 6

### Segment Results

- Low Frequency + Low Intensity: 13,681 employees
- Low Frequency + High Intensity: 8,977 employees
- High Frequency + Low Intensity: 2,229 employees
- High Frequency + High Intensity: 889 employees

### Interpretation

The largest group is Low Frequency + Low Intensity, indicating that many employees have only occasional short-duration non-working day presence.

The most critical review segment is High Frequency + High Intensity, containing 889 employees. These employees repeatedly appear on non-working days and also show substantial average TIC hours.

This segment may indicate repeated off-day workload, support dependency, project/client-driven weekend work, or potential work-life balance risk.

However, this should not be treated as a confirmed violation. These cases require business validation using project, shift, approval, and workload context.


## Location Distribution of High Frequency + High Intensity Employees

This section checks where the most critical employee segment is concentrated.

The segment `High Frequency + High Intensity` represents employees who:

- Were marked PRESENT on 5 or more unique non-working dates
- Had average TIC hours greater than or equal to 6 hours

The purpose is to identify whether repeated long-duration non-working day presence is concentrated in specific locations.

In [ ]:
high_freq_high_intensity_employees = employee_repetition[
    employee_repetition["employee_presence_segment"] == "High Frequency + High Intensity"
]

print("High Frequency + High Intensity employees:", high_freq_high_intensity_employees.shape[0])

high_freq_high_intensity_employees.head()

In [ ]:
high_freq_high_intensity_records = non_working_present[
    non_working_present["EMP_CODE"].isin(high_freq_high_intensity_employees["EMP_CODE"])
]

print("Records for High Frequency + High Intensity employees:", high_freq_high_intensity_records.shape[0])

high_freq_high_intensity_location = (
    high_freq_high_intensity_records["LOCATION"]
    .value_counts()
    .reset_index()
)

high_freq_high_intensity_location.columns = [
    "LOCATION",
    "high_frequency_high_intensity_record_count"
]

high_freq_high_intensity_location.head(15)

## Location Distribution of High Frequency + High Intensity Segment

The High Frequency + High Intensity segment contains 889 employees and 6,750 non-working day PRESENT records.

Location-wise, the highest record concentration appears in:

- Bengaluru: 1,894 records
- Mumbai: 1,195 records
- Pune: 1,095 records
- Kolkata: 1,021 records
- Hyderabad: 483 records
- Coimbatore: 330 records

This shows that the repeated long-duration non-working day presence pattern is most visible in larger delivery locations.

Mumbai and Kolkata remain important because they already showed higher TIC hour patterns in the earlier location-level analysis. Their presence in this segment further supports the observation that non-working day work in these locations is more likely to involve substantial campus hours.

Mysore has only 64 records in this segment, supporting the earlier conclusion that Mysore’s high non-working day PRESENT rate is mostly driven by short-duration presence rather than repeated high-intensity work.

In [ ]:
high_freq_high_intensity_unique_location = (
    high_freq_high_intensity_records
    .groupby("LOCATION")["EMP_CODE"]
    .nunique()
    .reset_index(name="unique_high_freq_high_intensity_employees")
    .sort_values("unique_high_freq_high_intensity_employees", ascending=False)
)

high_freq_high_intensity_unique_location.head(15)

In [ ]:
# Total unique employees with non-working day PRESENT records by location
location_unique_non_working_present_employees = (
    non_working_present
    .groupby("LOCATION")["EMP_CODE"]
    .nunique()
    .reset_index(name="unique_non_working_present_employees")
)

# Merge with unique high frequency + high intensity employees by location
high_freq_high_intensity_location_rate = (
    high_freq_high_intensity_unique_location
    .merge(
        location_unique_non_working_present_employees,
        on="LOCATION",
        how="left"
    )
)

# Calculate rate
high_freq_high_intensity_location_rate["high_freq_high_intensity_employee_rate_pct"] = (
    high_freq_high_intensity_location_rate["unique_high_freq_high_intensity_employees"] /
    high_freq_high_intensity_location_rate["unique_non_working_present_employees"] * 100
)

high_freq_high_intensity_location_rate.sort_values(
    "high_freq_high_intensity_employee_rate_pct",
    ascending=False
).head(15)

Although Noida and Ahmedabad show high percentages of High Frequency + High Intensity employees, their total employee counts are very small. Therefore, these rates may be unstable due to the small denominator effect.

For deeper analysis, locations should be prioritized based on both volume and rate.

Based on this principle, Mumbai, Kolkata, Pune, Bengaluru, and Coimbatore appear to be more meaningful focus areas.

- Bengaluru has the highest absolute number of High Frequency + High Intensity employees.
- Mumbai has both strong count and the highest meaningful rate among large locations.
- Kolkata also shows strong count and aligns with earlier high TIC-hour patterns.
- Pune has strong volume and should be included for comparison.
- Coimbatore has moderate count and a meaningful rate.
- Mysore should be treated separately because it shows high non-working day presence but mostly short-duration TIC hours.

## Career Band Distribution of High Frequency + High Intensity Employees

This section checks whether the High Frequency + High Intensity non-working day PRESENT pattern is concentrated in specific career bands.

Since the exact hierarchy or designation mapping of CAREER_BAND is not yet confirmed, CAREER_BAND is treated only as a categorical employee segment.

No seniority or role-based conclusion is made at this stage.

In [ ]:
career_band_high_freq_high_intensity = (
    high_freq_high_intensity_records
    .groupby("CAREER_BAND")["EMP_CODE"]
    .nunique()
    .reset_index(name="unique_high_freq_high_intensity_employees")
    .sort_values("unique_high_freq_high_intensity_employees", ascending=False)
)

career_band_high_freq_high_intensity

## Career Band View of High Frequency + High Intensity Employees

The High Frequency + High Intensity segment was analyzed by CAREER_BAND.

The highest raw counts appear in:

- GROUP ZZ: 365 employees
- GROUP Y1: 142 employees
- GROUP Y2: 84 employees
- BAND Q: 81 employees
- BAND P: 66 employees

This suggests that the repeated high-intensity non-working day presence pattern is more visible in certain career band categories.

However, the exact hierarchy or role mapping of CAREER_BAND has not yet been confirmed. Therefore, these results should be interpreted only as categorical segmentation and not as seniority or designation-level conclusions.

Also, raw counts may be influenced by the total number of employees in each career band. Therefore, a normalized career-band rate should be calculated before identifying which bands are proportionally more affected.

## Location and Career Band Cross-View

After identifying the High Frequency + High Intensity employee segment, the next step is to examine whether this segment is concentrated in specific location and career band combinations.

This helps identify whether repeated long-duration non-working day presence is broadly spread across the workforce or concentrated among particular employee categories within specific locations.

Since the exact hierarchy or role mapping of CAREER_BAND is not yet confirmed, CAREER_BAND is treated only as a categorical segment. No seniority or designation-level conclusion is made at this stage.

In [ ]:
location_career_band_high_freq_high_intensity = (
    high_freq_high_intensity_records
    .groupby(["LOCATION", "CAREER_BAND"])["EMP_CODE"]
    .nunique()
    .reset_index(name="unique_high_freq_high_intensity_employees")
    .sort_values("unique_high_freq_high_intensity_employees", ascending=False)
)

location_career_band_high_freq_high_intensity.head(20)

## Location and Career Band Rate Analysis

This section analyzes High Frequency + High Intensity employees across LOCATION and CAREER_BAND.

Raw counts alone can be misleading because large locations or large career bands naturally have more employees. Therefore, this analysis calculates both unique employee counts and normalized rates.

The rate is calculated as:

High Frequency + High Intensity Rate =
Unique High Frequency + High Intensity Employees /
Total Unique Non-working Day PRESENT Employees
within the same LOCATION and CAREER_BAND group

Since CAREER_BAND mapping is not yet confirmed, it is treated only as a categorical employee segment.

In [ ]:
# Numerator: unique High Frequency + High Intensity employees by LOCATION + CAREER_BAND
critical_loc_band = (
    high_freq_high_intensity_records
    .groupby(["LOCATION", "CAREER_BAND"])["EMP_CODE"]
    .nunique()
    .reset_index(name="unique_high_freq_high_intensity_employees")
)

# Denominator: total unique non-working day PRESENT employees by LOCATION + CAREER_BAND
base_loc_band = (
    non_working_present
    .groupby(["LOCATION", "CAREER_BAND"])["EMP_CODE"]
    .nunique()
    .reset_index(name="unique_non_working_present_employees")
)

# Merge numerator and denominator
loc_band_rate = critical_loc_band.merge(
    base_loc_band,
    on=["LOCATION", "CAREER_BAND"],
    how="left"
)

# Calculate rate
loc_band_rate["high_freq_high_intensity_rate_pct"] = (
    loc_band_rate["unique_high_freq_high_intensity_employees"] /
    loc_band_rate["unique_non_working_present_employees"] * 100
)

# Sort by unique critical employee count first
loc_band_rate.sort_values(
    "unique_high_freq_high_intensity_employees",
    ascending=False
).head(20)

## Location and Career Band Cross-View Interpretation

The Location × Career Band analysis shows that the High Frequency + High Intensity segment is not evenly distributed across all groups.

The strongest raw-count combinations are:

- Kolkata + GROUP ZZ: 125 employees
- Mumbai + GROUP ZZ: 108 employees
- Pune + GROUP ZZ: 82 employees
- Bengaluru + BAND Q: 71 employees
- Bengaluru + GROUP Y1: 46 employees
- Bengaluru + GROUP Y2: 36 employees

Kolkata and Mumbai continue to stand out because both locations showed substantial TIC hours earlier, and now GROUP ZZ within these locations also shows a meaningful concentration of High Frequency + High Intensity employees.

Pune + GROUP ZZ shows a particularly high rate of approximately 9.82%, which suggests that this location-career band combination may have a stronger proportional concentration of repeated high-intensity non-working day presence.

Bengaluru has the highest overall volume, but its High Frequency + High Intensity pattern appears to be spread across multiple career bands such as BAND Q, GROUP Y1, GROUP Y2, and BAND P.

Some combinations such as Kochi + GROUP ZZ show high rates, but the denominator is relatively small. Therefore, such results should be interpreted cautiously and not treated as strong conclusions without additional validation.

Since the exact hierarchy or role mapping of CAREER_BAND is not yet confirmed, these findings should be treated as categorical segmentation insights rather than seniority or designation-based conclusions.

## Location-Career Band Story

The cross-view shows that the High Frequency + High Intensity segment is not evenly distributed across all location-career band combinations.

The strongest raw-count combinations are:

- Kolkata + GROUP ZZ
- Mumbai + GROUP ZZ
- Pune + GROUP ZZ
- Bengaluru + BAND Q

This suggests that repeated long-duration non-working day presence may be concentrated in specific employee categories within certain locations.

However, since CAREER_BAND mapping is not yet confirmed, these results are treated as categorical segmentation insights rather than role or seniority conclusions.

## Chart 1: Top Location-Career Band Combinations by Unique Critical Employees

This chart shows the top LOCATION + CAREER_BAND combinations by the number of unique High Frequency + High Intensity employees.

This is a volume-based view.

It helps identify which location-career band combinations contribute the highest number of employees to the repeated high-intensity non-working day presence segment.

Since this is based on raw count, it should be interpreted together with the normalized rate analysis.

In [ ]:
import matplotlib.pyplot as plt

# Create readable label for each location-career band combination
loc_band_rate["location_career_band"] = (
    loc_band_rate["LOCATION"] + " | " + loc_band_rate["CAREER_BAND"]
)

# Select top 10 combinations by unique High Frequency + High Intensity employees
top_volume_loc_band = (
    loc_band_rate
    .sort_values("unique_high_freq_high_intensity_employees", ascending=False)
    .head(10)
)

# Plot horizontal bar chart
plt.figure(figsize=(12, 6))

plt.barh(
    top_volume_loc_band["location_career_band"],
    top_volume_loc_band["unique_high_freq_high_intensity_employees"]
)

plt.xlabel("Unique High Frequency + High Intensity Employees")
plt.ylabel("Location | Career Band")
plt.title("Top Location-Career Band Combinations by Unique Critical Employees")

plt.gca().invert_yaxis()

plt.show()

### Interpretation of Volume-Based Location-Career Band Chart

The volume-based chart shows that the largest High Frequency + High Intensity employee combinations are concentrated in:

- Kolkata + GROUP ZZ
- Mumbai + GROUP ZZ
- Pune + GROUP ZZ
- Bengaluru + BAND Q
- Bengaluru + GROUP Y1

This suggests that GROUP ZZ appears prominently in Kolkata, Mumbai, and Pune within the repeated high-intensity non-working day presence segment.

Bengaluru also contributes strongly, but the pattern appears more distributed across multiple career bands such as BAND Q, GROUP Y1, GROUP Y2, and BAND P.

This chart is a raw-count view, so it identifies where the highest number of unique critical employees are present. However, raw counts should be interpreted along with normalized rates because larger location-career band groups naturally have more employees.

In [ ]:
# Filter groups with meaningful denominator
loc_band_rate_filtered = loc_band_rate[
    loc_band_rate["unique_non_working_present_employees"] >= 100
].copy()

# Create readable label
loc_band_rate_filtered["location_career_band"] = (
    loc_band_rate_filtered["LOCATION"] + " | " + loc_band_rate_filtered["CAREER_BAND"]
)

# Top 10 by normalized rate
top_rate_loc_band = (
    loc_band_rate_filtered
    .sort_values("high_freq_high_intensity_rate_pct", ascending=False)
    .head(10)
)

plt.figure(figsize=(12, 6))

plt.barh(
    top_rate_loc_band["location_career_band"],
    top_rate_loc_band["high_freq_high_intensity_rate_pct"]
)

plt.xlabel("High Frequency + High Intensity Rate (%)")
plt.ylabel("Location | Career Band")
plt.title("Top Location-Career Band Combinations by Critical Employee Rate")

plt.gca().invert_yaxis()

plt.show()

### Interpretation of Rate-Based Location-Career Band Chart

The rate-based chart shows which LOCATION + CAREER_BAND combinations have the strongest proportional concentration of High Frequency + High Intensity employees.

Unlike the volume chart, this view normalizes the number of critical employees by the total number of non-working day PRESENT employees within the same LOCATION + CAREER_BAND group.

Key observations:

- **Pune + GROUP ZZ** shows the highest rate among meaningful groups, indicating a strong proportional concentration of repeated high-intensity non-working day presence.
- **Mumbai** appears across multiple career bands such as GROUP Y2, GROUP ZZ, GROUP Y1, and GROUP Y3, suggesting that Mumbai’s pattern may be spread across multiple employee categories.
- **Kolkata + GROUP ZZ** remains important because it appeared strongly in both the volume-based and rate-based views.
- **Bengaluru** appears in the rate chart through BAND P and BAND Q, but Bengaluru’s strongest story remains its overall volume rather than the highest proportional rate.
- Some combinations such as Bhubaneswar + GROUP ZZ should be interpreted cautiously if their base size is relatively small.

Overall, the rate chart supports the idea that repeated high-intensity non-working day presence is not evenly distributed. It is more prominent in specific location-career band combinations, especially Pune + GROUP ZZ, Mumbai groups, and Kolkata + GROUP ZZ.

### Location-Career Band Cross-View Summary

The combined volume and rate views suggest that the High Frequency + High Intensity pattern is concentrated in specific location-career band combinations.

Kolkata + GROUP ZZ and Mumbai + GROUP ZZ are strong from a volume perspective, while Pune + GROUP ZZ shows the strongest proportional rate among meaningful groups.

Mumbai appears across multiple career bands, indicating that the repeated high-intensity non-working day presence pattern may be broader within that location.

Since CAREER_BAND mapping is not yet confirmed, these findings should be treated as segmentation-level observations rather than role or seniority conclusions.

## Final Outcome

The analysis identifies non-working day PRESENT records as a meaningful operational pattern in the attendance dataset.

These records are not merely status labels; many contain measurable TIC, TOF, and TOTAL hour values. The pattern differs significantly by location, employee repetition, and career band category.

Bengaluru contributes the highest absolute volume of such records, while Mumbai and Kolkata show stronger full-day or near full-day non-working presence patterns. Mysore shows a different pattern with high frequency but mostly short-duration presence.

High-hour records above 24 hours exist but represent only 0.71% of non-working day PRESENT records, suggesting that extreme values are rare and should be treated as validation cases rather than the main driver of the pattern.

Employee-level segmentation identified 889 High Frequency + High Intensity employees, representing the most review-worthy group for repeated and substantial non-working day presence.

Location-career band analysis further shows that the pattern is concentrated in specific combinations such as Kolkata + GROUP ZZ, Mumbai + GROUP ZZ, Pune + GROUP ZZ, and Bengaluru + BAND Q.

Overall, this pattern may reflect business-driven weekend/holiday work, support dependency, project/client requirements, or operational workload concentration. However, final conclusions require validation using business context such as project schedules, approvals, shift rosters, and career band mapping.

| Point | Status |
|---|---|
| Mumbai/Kolkata/Pune GROUP ZZ pattern | Pattern found, business validation pending |
| 889 High Frequency + High Intensity employees | Segment identified, review pending |
| High-hour >24 cases | 446 cases identified, validation pending |
| Mysore short-duration pattern | Strongly supported by data |
| Do not call violation | Caution/principle, not an analysis output |

## Workday Comparison for High Frequency + High Intensity Employees

This section checks whether employees who repeatedly appear on non-working days with high average TIC hours also show high campus presence on normal WORKDAY records.

The purpose is to understand whether the pattern may indicate sustained workload across both workdays and off-days, or whether it is more specific to weekend/holiday work.

Since employees may have multiple project-level records for the same EMP_CODE and ATTENDANCE_DATE, the workday analysis is first aggregated at employee-date level using maximum TIC hours to reduce the risk of double-counting.

In [ ]:
# Get EMP_CODE list for High Frequency + High Intensity employees
high_freq_high_intensity_employee_list = high_freq_high_intensity_employees["EMP_CODE"]

# Filter WORKDAY records for these employees
high_freq_high_intensity_workday_records = df[
    (df["EMP_CODE"].isin(high_freq_high_intensity_employee_list)) &
    (df["WEEKOFF_HOLIDAY"] == "WORKDAY")
]

print("WORKDAY records for High Frequency + High Intensity employees:",
      high_freq_high_intensity_workday_records.shape[0])

high_freq_high_intensity_workday_records.head()

### Workday Context for High Frequency + High Intensity Employees

Before comparing TIC hours on workdays, the workday REMARKS distribution is analyzed.

This is important because TIC hours may be zero for valid work contexts such as Client Office, Home, Business Travel, or Training. Therefore, workday behavior should be interpreted using both TIC hours and REMARKS.

In [ ]:
workday_remarks_for_critical_employees = (
    high_freq_high_intensity_workday_records["REMARKS"]
    .value_counts(dropna=False)
    .reset_index()
)

workday_remarks_for_critical_employees.columns = [
    "REMARKS",
    "workday_record_count"
]

workday_remarks_for_critical_employees.head(15)


### Workday Context Interpretation

For the 889 High Frequency + High Intensity employees, the majority of WORKDAY records are marked as PRESENT.

Key observations:

- PRESENT: 31,176 records (~74.44%)
- Home: 5,386 records (~12.86%)
- FULL DAY LEAVE: 2,663 records (~6.36%)
- Wipro Office: 1,632 records (~3.90%)
- Client Office: 469 records (~1.12%)

This indicates that the employees who repeatedly appear on non-working days with high TIC hours also have a large number of normal workday records marked as PRESENT.

However, workday remarks such as Home, Client Office, Leave, and OOD show that not all workday records represent campus presence. Therefore, TIC-based workday comparison should be performed carefully, ideally after separating valid workday presence records from leave or non-campus contexts.

## Workday TIC Pattern for High Frequency + High Intensity Employees

This section analyzes normal WORKDAY + PRESENT records for employees who belong to the High Frequency + High Intensity non-working day segment.

The objective is to check whether these employees also show substantial campus presence on regular workdays.

Since the same employee and attendance date may have multiple project-level records, TIC hours are aggregated at EMP_CODE + ATTENDANCE_DATE level using maximum TIC hours to reduce the risk of double-counting.

In [ ]:
# Filter WORKDAY + PRESENT records for High Frequency + High Intensity employees
high_freq_high_intensity_workday_present = df[
    (df["EMP_CODE"].isin(high_freq_high_intensity_employee_list)) &
    (df["WEEKOFF_HOLIDAY"] == "WORKDAY") &
    (df["REMARKS"] == "PRESENT")
]

print("WORKDAY + PRESENT records for High Frequency + High Intensity employees:",
      high_freq_high_intensity_workday_present.shape[0])


# Aggregate to employee-date level using max TIC to avoid double-counting
workday_present_daily = (
    high_freq_high_intensity_workday_present
    .groupby(["EMP_CODE", "ATTENDANCE_DATE"], as_index=False)
    .agg(
        max_workday_tic_hours=("TIC_HOURS_NEW", "max"),
        max_workday_total_hours=("TOTAL_HOURS", "max")
    )
)

print("Unique employee-date WORKDAY + PRESENT records:",
      workday_present_daily.shape[0])


# Employee-level workday TIC summary
workday_present_employee_summary = (
    workday_present_daily
    .groupby("EMP_CODE", as_index=False)
    .agg(
        workday_present_dates=("ATTENDANCE_DATE", "nunique"),
        avg_workday_tic_hours=("max_workday_tic_hours", "mean"),
        median_workday_tic_hours=("max_workday_tic_hours", "median"),
        max_workday_tic_hours=("max_workday_tic_hours", "max"),
        workday_6plus_tic_dates=("max_workday_tic_hours", lambda x: (x >= 6).sum())
    )
)

workday_present_employee_summary.head(20)

In [ ]:
workday_present_employee_summary[
    [
        "workday_present_dates",
        "avg_workday_tic_hours",
        "median_workday_tic_hours",
        "max_workday_tic_hours",
        "workday_6plus_tic_dates"
    ]
].describe()

## Workday TIC Pattern Interpretation

The 889 High Frequency + High Intensity employees were also analyzed on their normal WORKDAY + PRESENT records.

The results show that these employees have substantial campus presence on regular workdays as well:

- Average WORKDAY + PRESENT dates per employee: approximately 33 days
- Median WORKDAY + PRESENT dates per employee: 35 days
- Average workday TIC hours: approximately 9.59 hours
- Median workday TIC hours: approximately 9.86 hours
- Average number of workday dates with TIC >= 6 hours: approximately 30 days
- Median number of workday dates with TIC >= 6 hours: 32 days

This indicates that the High Frequency + High Intensity employees are not only repeatedly present on non-working days, but also show consistently high campus presence on normal workdays.

This strengthens the interpretation that this group may represent sustained workload, project/client dependency, support responsibility, or operational pressure.

However, this should not be treated as confirmed overwork or policy violation without validating business context such as project allocation, shift schedules, approvals, and client support requirements.

## Final Conclusion: Non-Working Day PRESENT Analysis

### Overall Finding

The analysis identified a meaningful pattern where records are marked as `PRESENT` on non-working days such as `WEEK-OFF` and `HOLIDAY`.

A total of **62,969 records** were found where employees were marked `PRESENT` on non-working days. These records also contained measurable `TIC_HOURS_NEW`, `TOF_HOURS_NEW`, and `TOTAL_HOURS`, which suggests that this is not just a status-label pattern but may represent actual campus/ODC presence.

However, all findings are interpreted cautiously because the dataset is at record level and employees may have multiple project-level records for the same date.

---

### Key Findings

#### 1. Non-working Day PRESENT Records Have Actual Hours

The non-working day PRESENT records showed:

- Average TIC hours: approximately **4.52 hours**
- Median TIC hours: approximately **2.71 hours**
- 75th percentile TIC hours: approximately **8.27 hours**
- Maximum TIC hours: approximately **42.33 hours**

This indicates that many non-working day PRESENT records include actual time spent in campus or ODC/floor areas.

---

#### 2. Location-Level Pattern Is Not Uniform

The pattern differs significantly across locations.

- **Bengaluru** has the highest absolute volume of non-working day PRESENT records.
- **Mumbai and Kolkata** show stronger full-day or near full-day non-working day presence patterns based on higher average and median TIC hours.
- **Mysore** shows a different pattern: high non-working day PRESENT rate but mostly short-duration TIC hours.
- **Coimbatore and Bengaluru** show mixed-duration behavior.

This means non-working day presence should not be interpreted uniformly across all locations.

---

#### 3. High-Hour Records Exist But Are Rare

High-hour records were defined as records where `TIC_HOURS_NEW`, `TOF_HOURS_NEW`, or `TOTAL_HOURS` exceeded 24 hours.

Results:

- High-hour records: **446**
- Percentage of non-working day PRESENT records: **0.71%**

This suggests that extreme high-hour cases exist but do not appear to be the main driver of the overall non-working day PRESENT pattern.

These high-hour cases should be treated as validation cases because they may be caused by:

- Multi-project allocation effects
- Overnight shift logic
- System calculation behavior
- Repeated allocation records
- Exceptional support or project work

---

#### 4. Employee-Level Segmentation Identified a Critical Review Group

Employees were segmented using Frequency and Intensity.

- **Frequency** = number of unique non-working dates where the employee was marked PRESENT
- **Intensity** = average TIC hours during non-working day PRESENT records

Exploratory thresholds used:

- High Frequency = 5 or more unique non-working PRESENT dates
- High Intensity = average TIC hours greater than or equal to 6 hours

Segment results:

- Low Frequency + Low Intensity: **13,681 employees**
- Low Frequency + High Intensity: **8,977 employees**
- High Frequency + Low Intensity: **2,229 employees**
- High Frequency + High Intensity: **889 employees**

The **889 High Frequency + High Intensity employees** represent the most review-worthy segment because they repeatedly appear on non-working days and also show substantial average campus presence.

---

#### 5. High Frequency + High Intensity Employees Also Show High Workday Presence

For the 889 High Frequency + High Intensity employees, their normal `WORKDAY + PRESENT` records were analyzed.

Key results:

- Average WORKDAY + PRESENT dates per employee: approximately **33 days**
- Median WORKDAY + PRESENT dates per employee: **35 days**
- Average workday TIC hours: approximately **9.59 hours**
- Median workday TIC hours: approximately **9.86 hours**
- Average number of workday dates with TIC >= 6 hours: approximately **30 days**
- Median number of workday dates with TIC >= 6 hours: **32 days**

This suggests that these employees are not only repeatedly present on non-working days, but also show substantial campus presence on regular workdays.

This may indicate sustained workload, project/client dependency, support responsibility, or operational pressure. However, this should not be treated as confirmed overwork without business validation.

---

#### 6. Location and Career Band Cross-View Shows Specific Concentrations

The High Frequency + High Intensity segment was also analyzed by `LOCATION` and `CAREER_BAND`.

Strong raw-count combinations include:

- Kolkata + GROUP ZZ
- Mumbai + GROUP ZZ
- Pune + GROUP ZZ
- Bengaluru + BAND Q
- Bengaluru + GROUP Y1

Rate-based analysis also showed strong proportional concentration in:

- Pune + GROUP ZZ
- Mumbai + GROUP Y2
- Kolkata + GROUP ZZ
- Mumbai + GROUP ZZ
- Bengaluru + BAND P / BAND Q

This suggests that repeated high-intensity non-working day presence is not evenly distributed. It appears more concentrated in specific location-career band combinations.

However, since the exact hierarchy or role mapping of `CAREER_BAND` is not yet confirmed, these findings should be treated as categorical segmentation insights and not as seniority or designation-level conclusions.

---

### Business Interpretation

The non-working day PRESENT pattern may represent:

- Weekend or holiday project work
- Client support requirements
- Shift-based operations
- Deployment or maintenance activity
- Business-critical work
- Multi-project allocation effects
- Data quality or classification issues

The pattern should not be directly treated as a violation or anomaly.

Instead, it should be treated as a **review-worthy operational pattern**.

---

### Business Implications

#### Positive Signal

Non-working day presence may indicate:

- Operational commitment
- Business continuity
- Client/project support
- Critical work being handled outside regular workdays

#### Potential Workforce Risk

If repeated full-day work on week-offs or holidays is sustained over time, it may create:

- Work-life balance concerns
- Employee fatigue
- Motivation decline
- Burnout risk
- Workload concentration risk
- Dependency on specific employees or teams

#### Data Validation Need

Some records require validation before final interpretation, especially:

- High-hour records greater than 24 hours
- Multi-project employee-date records
- Specific location-career band concentrations
- High Frequency + High Intensity employee segment
- Career band mapping

---

### Final Outcome

The analysis shows that non-working day PRESENT records form a meaningful pattern in the dataset.

This pattern is supported by actual campus and ODC/floor hour values and varies across locations, employees, and career band categories.

The most important outcome is the identification of **889 High Frequency + High Intensity employees**, who repeatedly appear on non-working days and also show substantial campus presence on regular workdays.

This group should be considered a priority review segment for business validation, workload understanding, and possible support/operational planning.

No final compliance or violation conclusion should be made until the findings are validated using:

- Project context
- Shift schedules
- Weekend/holiday work approvals
- Client support requirements
- Career band mapping
- Exception and leave data

---

### Recommended Next Steps

1. Validate Mumbai, Kolkata, Pune, and Bengaluru patterns with business/project context.
2. Review the 889 High Frequency + High Intensity employees for repeated off-day workload.
3. Validate high-hour records above 24 hours separately as data/system/shift review cases.
4. Treat Mysore separately as a short-duration presence pattern rather than a full-day off-day workload pattern.
5. Confirm career band mapping before making role or seniority-level interpretations.
6. Move to the next notebook for broader WORKDAY pattern analysis.